In [1]:
# ABC ka matlab hai Abstract Base Class.

from abc import ABC, abstractmethod

In [77]:
# @abstractmethod batata hai ki child classes ko invoke() method implement karna hi padega.
# pass ka matlab hai ki yahan abhi koi implementation nahi hai.

class Runnable(ABC):
    
    @abstractmethod
    def invoke(self, input_data):
        pass

In [78]:
# Ye ek fake LLM hai. Ye tumhare prompt ko actually samajhkar answer generate nahi karta. Ye list mein se random response select karta hai.

import random

class NakliLLM(Runnable):
    
    def __init__(self):
        print('LLM created')
        
    def invoke(self, prompt):
        response_list = [   
            'delhi is the capital of india',
            'IPL is a cricket league',
            'AI stands for artificial intellegence'
        ]
        
        return {'response':random.choice(response_list)}
    
# prompt parameter yahan use nahi ho raha. Isliye cricket, India ya AI ke baare mein kuch bhi poochho, output in teen responses mein se random hoga.
# Tumhare code mein predict() method bhi hai, jo isi tarah random response deta hai.
    
    def predict(self, prompt):
        
        response_list = [
            'delhi is the capital of india',
            'IPL is a cricket league',
            'AI stands for artificial intellegence'
        ]
        
        return {'response':random.choice(response_list)}

In [95]:
# Iska kaam hai template ke andar variables ki values daal kar prompt banana.

class NakliPromptTemplate(Runnable):
    
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables
        
# Yahan format(**input_dict) dictionary ki values ko template ke placeholders mein daalta hai.

    def invoke(self, input_dict):
        return self.template.format(**input_dict)
    
    def format(self, input_dict):
        return self.template.format(**input_dict)

In [96]:
# NakliStrOutputParser — output ko string mein badalna

class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass
# Parser us dictionary se sirf response ki value nikaal leta hai.

    def invoke(self, input_data):
        return input_data['response']

In [81]:
# Iska kaam hai multiple components ko ek pipeline mein chalana.

class RunnableConnector(Runnable):
    
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list
        
    def invoke(self, input_data):
        
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
            
        return input_data

In [82]:
template = NakliPromptTemplate(
    template='write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

In [83]:
llm = NakliLLM()

LLM created


In [84]:
parser = NakliStrOutputParser()

In [85]:
chain = RunnableConnector([template, llm, parser])

In [86]:
chain.invoke({'length':'long', 'topic':'india'})

'delhi is the capital of india'

In [104]:
template1 = NakliPromptTemplate(
    template='write a joke about {topic}',
    input_variables=['topic']
)

In [105]:
template2 = NakliPromptTemplate(
    template='explain the following joke {response}',
    input_variables=['response']
)

In [106]:
llm = NakliLLM()

LLM created


In [107]:
parser = NakliStrOutputParser()

In [108]:
chain1 = RunnableConnector([template1, llm])

In [109]:
chain2 = RunnableConnector([template2, llm, parser])

In [110]:
final_chain = RunnableConnector([chain1, chain2])

In [112]:
final_chain.invoke({'topic':'cricket'})

'AI stands for artificial intellegence'